# Desafio MNIST: uma rede densa que lê dígitos

**Missão:** treinar uma rede neural que olha um dígito escrito à mão e diz
qual é, de 0 a 9.

**Como funciona** (em duplas):

- Complete os `____` no código, de cima para baixo.
- Antes de cada treino, escreva a sua **previsão**. Depois, explique o que viu.
- Mude **uma coisa por vez**. Não é competição de acurácia: vale entender o que muda.

✏️ = você escreve (dois cliques na célula) &nbsp;·&nbsp; 📦 = caixa de referência

Antes de começar: **Arquivo → Salvar uma cópia no Drive**.

In [ ]:
# Preparação: bibliotecas e funções prontas. Só rode, não precisa mexer.
import time
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers

keras.utils.set_random_seed(42)


def mostrar_digitos(imagens, rotulos, quantas=10):
    # Mostra as primeiras imagens, com o rótulo de cada uma em cima
    plt.figure(figsize=(12, 2.4))
    for i in range(quantas):
        plt.subplot(1, quantas, i + 1)
        plt.imshow(imagens[i], cmap="gray")
        plt.title(rotulos[i])
        plt.axis("off")
    plt.show()


def plotar_curvas(historico):
    # Perda e acurácia, no treino e na validação, ao longo das épocas
    plt.figure(figsize=(11, 4))

    plt.subplot(1, 2, 1)
    plt.plot(historico.history["loss"], label="treino")
    plt.plot(historico.history["val_loss"], label="validação")
    plt.xlabel("Época")
    plt.ylabel("Perda")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(historico.history["accuracy"], label="treino")
    plt.plot(historico.history["val_accuracy"], label="validação")
    plt.xlabel("Época")
    plt.ylabel("Acurácia")
    plt.legend()

    plt.show()


def mostrar_previsoes(rede, entradas, quais="erros", quantas=8):
    # Mostra imagens do teste em que a rede acertou ou errou
    respostas = rede.predict(entradas, verbose=0).argmax(axis=1)
    if quais == "erros":
        indices = np.where(respostas != digitos_teste)[0]
    else:
        indices = np.where(respostas == digitos_teste)[0]

    plt.figure(figsize=(12, 2.6))
    for posicao, indice in enumerate(indices[:quantas]):
        plt.subplot(1, quantas, posicao + 1)
        plt.imshow(imagens_teste[indice], cmap="gray")
        plt.title(f"era {digitos_teste[indice]}\ndisse {respostas[indice]}", fontsize=10)
        plt.axis("off")
    plt.show()
    print(f"A rede {'errou' if quais == 'erros' else 'acertou'} {len(indices)} de 10.000 imagens do teste.")


def treinar_e_avaliar(rede, entradas_treino, entradas_teste, epocas):
    # Compila (adam + sparse_categorical_crossentropy), treina e mede tudo de uma vez
    rede.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
                 metrics=["accuracy"])
    inicio = time.time()
    historico = rede.fit(entradas_treino, digitos_treino, epochs=epocas,
                         validation_split=0.1, verbose=0)
    segundos = time.time() - inicio
    _, acuracia_teste = rede.evaluate(entradas_teste, digitos_teste, verbose=0)

    print(f"Acurácia treino:    {historico.history['accuracy'][-1]:.1%}")
    print(f"Acurácia validação: {historico.history['val_accuracy'][-1]:.1%}")
    print(f"Acurácia teste:     {acuracia_teste:.1%}")
    print(f"Parâmetros: {rede.count_params():,}  ·  Tempo: {segundos:.0f} s")
    return historico

## 0. O problema

Entra uma imagem de 28 × 28 pixels com um dígito escrito à mão. Sai o dígito
que ela mostra.

✏️ **Responda em uma linha cada:**

1. O que entra no modelo? →
2. O que queremos que saia? →
3. Quantas classes existem? →
4. O que quer dizer "acertar"? →

In [ ]:
# O MNIST vem dentro do Keras: 60.000 imagens para treinar e 10.000 para testar
(imagens_treino, digitos_treino), (imagens_teste, digitos_teste) = keras.datasets.mnist.load_data()

print(f"Treino: {imagens_treino.shape} imagens, {digitos_treino.shape} rótulos")
print(f"Teste:  {imagens_teste.shape} imagens")
print(f"Os pixels vão de {imagens_treino.min()} a {imagens_treino.max()}")

mostrar_digitos(imagens_treino, digitos_treino)

✏️ **Interprete o que apareceu:**

- Quantas imagens de treino existem, e que formato cada uma tem? →
- O que significam os pixels 0 e 255? →

## 1. Preparar os dados

Os pixels vão de 0 a 255. Vamos levá-los para o intervalo de 0 a 1.

✏️ Por que valores pequenos ajudam a rede? →

In [ ]:
# Divida cada pixel por um número que leve o 255 a 1
imagens_treino = imagens_treino / ____
imagens_teste = imagens_teste / ____

In [ ]:
if 0.9 < imagens_treino.max() <= 1.0:
    print("✅ Pixels entre 0 e 1.")
else:
    print(f"❌ O maior pixel agora é {imagens_treino.max():.4f}.")
    print("   Rodou esta célula duas vezes? Volte e rode a célula que carrega os dados.")

## 2. A rede densa

Uma camada `Dense` espera uma **fila** de números. Cada imagem é uma
**tabela** de 28 × 28. Por isso o primeiro passo é enfileirar os pixels
(`Flatten`).

✏️ Se enfileirarmos todos os pixels, quantos números tem uma imagem?

In [ ]:
valores_por_imagem = ____ * ____
print(valores_por_imagem)

📦 **Referência.** Exemplos de *outro* problema, com 3 classes:

```python
layers.Flatten()                          # tabela vira fila
layers.Dense(64, activation="relu")       # camada escondida
layers.Dense(3, activation="softmax")     # saída: um neurônio por classe
```

✏️ **Antes de preencher:**

- Quantos neurônios a camada de saída precisa ter, e por quê? →
- Qual ativação vai no meio? E na saída (10 classes)? →

In [ ]:
rede_densa = keras.Sequential([
    keras.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(____, activation="____"),   # camada escondida
    layers.Dense(____, activation="____"),   # saída
])
rede_densa.summary()

In [ ]:
saida = rede_densa.layers[-1]
if saida.units == 10 and saida.activation.__name__ == "softmax":
    print("✅ Saída com 10 neurônios e softmax.")
else:
    print("❌ A saída precisa de um neurônio por classe e da ativação que devolve probabilidades.")

## 3. Compilar

Falta dizer **como** a rede aprende. Os rótulos são números inteiros de 0 a 9.

📦 **Referência:** `optimizer="adam"` é um bom padrão. Para rótulos inteiros
com várias classes, a perda é `"sparse_categorical_crossentropy"`.

In [ ]:
rede_densa.compile(
    optimizer="____",
    loss="____",
    metrics=["accuracy"],
)

## 4. Primeiro treino

✏️ **Antes de rodar, escreva:**

- Que acurácia você espera? →
- Quantas épocas acha que precisa? →
- A rede densa vai reconhecer bem os dígitos? →

In [ ]:
# Use poucas épocas: entre 3 e 8
historico_denso = rede_densa.fit(
    imagens_treino,
    digitos_treino,
    epochs=____,
    validation_split=0.1,
)

In [ ]:
plotar_curvas(historico_denso)

✏️ **Olhando as curvas:**

- O modelo está aprendendo? →
- Há sinal de overfitting (treino melhora, validação não)? →
- E se treinássemos por 100 épocas? →

## 5. Avaliar no teste

📦 **Referência:** `rede.evaluate(entradas, rótulos, verbose=0)` devolve a
perda e a acurácia.

In [ ]:
perda_teste, acuracia_teste = rede_densa.evaluate(____, ____, verbose=0)
print(f"Acurácia no teste: {acuracia_teste:.1%}")

In [ ]:
mostrar_previsoes(rede_densa, imagens_teste, quais="acertos")
mostrar_previsoes(rede_densa, imagens_teste, quais="erros")

✏️ **Escolha 3 erros:**

- Eles parecem compreensíveis para uma pessoa? →
- Quais dígitos foram confundidos? →
- Existe algum padrão? →

## 6. Três experimentos

Mude **uma coisa por vez** e preencha a tabela (dois cliques nela). Escreva
a previsão **antes** de rodar.

| Experimento | O que mudou | Previsão | Acurácia treino | Acurácia validação | Acurácia teste | Conclusão |
|---|---|---|---|---|---|---|
| base | nada (a rede acima) | | | | | |
| 1 | | | | | | |
| 2 | | | | | | |
| 3 | | | | | | |

### Experimento 1: o número de neurônios

✏️ Previsão: com mais (ou menos) neurônios na camada escondida, a acurácia vai... →

In [ ]:
rede_exp1 = keras.Sequential([
    keras.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(____, activation="relu"),    # <- só isto muda
    layers.Dense(10, activation="softmax"),
])
historico_exp1 = treinar_e_avaliar(rede_exp1, imagens_treino, imagens_teste, epocas=____)

### Experimento 2: uma segunda camada escondida

✏️ Previsão: uma camada a mais vai ajudar? →

In [ ]:
rede_exp2 = keras.Sequential([
    keras.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(____, activation="relu"),    # <- a camada nova
    layers.Dense(10, activation="softmax"),
])
historico_exp2 = treinar_e_avaliar(rede_exp2, imagens_treino, imagens_teste, epocas=____)

### Experimento 3: o número de épocas

✏️ Previsão: com bem mais épocas, treino e validação vão... →

In [ ]:
rede_exp3 = keras.Sequential([
    keras.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax"),
])
historico_exp3 = treinar_e_avaliar(rede_exp3, imagens_treino, imagens_teste, epocas=____)   # <- só isto muda
plotar_curvas(historico_exp3)

## 7. O que a rede densa não vê

A rede densa recebe a imagem **depois de enfileirada**. Cada pixel vira uma
entrada solta, e quem era vizinho de quem não aparece em lugar nenhum.

O teste: **embaralhar os pixels** de todas as imagens, sempre na mesma ordem.
Uma pessoa não reconhece mais nada.

✏️ Previsão: a acurácia da rede densa nas imagens embaralhadas vai cair
muito, um pouco, ou nada? →

In [ ]:
# Uma ordem aleatória, a mesma para todas as imagens
ordem = np.random.RandomState(0).permutation(28 * 28)
embaralhadas_treino = imagens_treino.reshape(-1, 784)[:, ordem].reshape(-1, 28, 28)
embaralhadas_teste = imagens_teste.reshape(-1, 784)[:, ordem].reshape(-1, 28, 28)

mostrar_digitos(embaralhadas_treino, digitos_treino, quantas=6)

# A mesma arquitetura da sua rede, com pesos novos
rede_embaralhada = keras.models.clone_model(rede_densa)
historico_embaralhado = treinar_e_avaliar(rede_embaralhada, embaralhadas_treino,
                                          embaralhadas_teste, epocas=5)

✏️ **O que isso mostra?** A rede densa usa a vizinhança dos pixels? →

---

### Para fechar

Se a rede densa nem nota o embaralhamento, falta a ela olhar **padrões
locais**: bordas, traços, curvas. É isso que a **convolução** faz, e é o
assunto da **Aula 8**: lá você treina uma rede convolucional no mesmo MNIST e
compara com esta.

✏️ **Anote para a Aula 8:**

- Acurácia de teste da sua rede densa (base): →
- Número de parâmetros dela: →

✏️ **Reflexão (uma linha cada):**

1. Qual experimento deu um resultado diferente do que você previa? →
2. Cite algo que você sabe agora porque **viu acontecer**, e não porque leu. →